# XAI-ED — Notebook 2: Model Training, Evaluation & Statistical Comparison

This notebook covers:
- Training all 4 classifiers (LogReg, Random Forest, XGBoost, LightGBM)
- Comprehensive metric evaluation (7 metrics)
- 5-fold stratified cross-validation with mean ± std
- ROC curves and calibration (reliability) diagrams
- McNemar's test for statistical significance between models
- Learning curves for overfitting analysis

**Run `scripts/run_all.py` first** to generate persisted models and metrics.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.metrics import roc_curve

from src.config import FEATURE_COLUMNS, TARGET_COLUMN
from src.data_loader import load_dataset
from src.train_model import train, build_model
from src.evaluate import (
    evaluate_binary, cross_validate_model,
    compare_models_mcnemar, get_calibration_data
)

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
sns.set_theme(style='darkgrid')
MODEL_LABELS = {'logreg': 'Logistic Regression', 'rf': 'Random Forest',
                'xgb': 'XGBoost', 'lgbm': 'LightGBM'}
MODEL_COLORS = {'logreg': '#42a5f5', 'rf': '#66bb6a', 'xgb': '#ffa726', 'lgbm': '#ab47bc'}
print('Libraries loaded.')

In [ ]:
X, y = load_dataset('../data/student_data.csv')
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Train: {len(X_train)} | Test: {len(X_test)} | Mastery rate: {y.mean():.3f}')

## 1. Train All 4 Models

In [ ]:
MODEL_NAMES = ['logreg', 'rf', 'xgb', 'lgbm']
trained_models = {}

for name in MODEL_NAMES:
    print(f'Training {name}...', end=' ')
    trained = train(name, X_train, y_train)
    trained_models[name] = trained.pipeline
    print('done')

print('\nAll models trained.')

## 2. Evaluation Metrics

In [ ]:
rows = []
all_metrics = {}
for name in MODEL_NAMES:
    m = evaluate_binary(trained_models[name], X_test, y_test)
    all_metrics[name] = m
    rows.append({
        'Model':      MODEL_LABELS[name],
        'Accuracy':   round(m['accuracy'],    3),
        'Precision':  round(m['precision'],   3),
        'Recall':     round(m['recall'],      3),
        'F1':         round(m['f1'],          3),
        'ROC-AUC':    round(m['roc_auc'],     3),
        'Brier':      round(m['brier_score'], 3),
        'MCC':        round(m['mcc'],         3),
    })

metrics_df = pd.DataFrame(rows)
print('Test-Set Metrics')
print(metrics_df.to_string(index=False))

## 3. Cross-Validation

In [ ]:
cv_results = {}
for name in MODEL_NAMES:
    print(f'  CV for {name}...')
    cv = cross_validate_model(
        model_factory=lambda mn=name: build_model(mn, y_train),
        X=X_train, y=y_train, n_splits=5,
    )
    cv_results[name] = cv

cv_rows = []
for name in MODEL_NAMES:
    cv = cv_results[name]
    cv_rows.append({
        'Model':          MODEL_LABELS[name],
        'CV AUC':         f"{cv['roc_auc']['mean']:.4f} ± {cv['roc_auc']['std']:.4f}",
        'CV F1':          f"{cv['f1']['mean']:.4f} ± {cv['f1']['std']:.4f}",
        'CV Accuracy':    f"{cv['accuracy']['mean']:.4f} ± {cv['accuracy']['std']:.4f}",
        'CV MCC':         f"{cv['mcc']['mean']:.4f} ± {cv['mcc']['std']:.4f}",
    })

cv_df = pd.DataFrame(cv_rows)
print('\n5-Fold Cross-Validation Results (mean ± std)')
print(cv_df.to_string(index=False))

## 4. ROC Curves & Calibration Diagrams

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for name in MODEL_NAMES:
    model = trained_models[name]
    color = MODEL_COLORS[name]
    label = MODEL_LABELS[name]
    auc   = all_metrics[name]['roc_auc']

    # ROC
    proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    axes[0].plot(fpr, tpr, label=f'{label} (AUC={auc:.3f})', color=color, lw=2)

    # Calibration
    cal = get_calibration_data(model, X_test, y_test)
    axes[1].plot(
        cal['mean_predicted_value'],
        cal['fraction_of_positives'],
        label=label, marker='o', color=color, lw=2
    )

# ROC diagonal
axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves', fontweight='bold')
axes[0].legend()

# Calibration diagonal
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Perfect calibration')
axes[1].set_xlabel('Mean Predicted Probability')
axes[1].set_ylabel('Fraction of Positives')
axes[1].set_title('Calibration Curves (Reliability Diagram)', fontweight='bold')
axes[1].legend()

plt.suptitle('Model Comparison: ROC + Calibration', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/roc_calibration.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. McNemar's Significance Tests

In [ ]:
mc_rows = []
for i in range(len(MODEL_NAMES)):
    for j in range(i + 1, len(MODEL_NAMES)):
        a, b = MODEL_NAMES[i], MODEL_NAMES[j]
        result = compare_models_mcnemar(trained_models[a], trained_models[b], X_test, y_test)
        mc_rows.append({
            'Model A': MODEL_LABELS[a],
            'Model B': MODEL_LABELS[b],
            'n01 (A wrong, B correct)': result['n01'],
            'n10 (A correct, B wrong)': result['n10'],
            'χ² statistic': result['statistic'],
            'p-value': result['p_value'],
            'Significant (α=0.05)': '✅' if result['significant'] else '❌',
        })

mc_df = pd.DataFrame(mc_rows)
print('McNemar Pairwise Significance Tests')
print(mc_df.to_string(index=False))

## 6. Learning Curves (Overfitting Analysis)

In [ ]:
from sklearn.metrics import make_scorer, roc_auc_score

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
train_sizes = np.linspace(0.1, 1.0, 8)
auc_scorer  = make_scorer(roc_auc_score, needs_proba=True)

for i, name in enumerate(MODEL_NAMES):
    model = build_model(name, y_train)
    sizes, tr_scores, val_scores = learning_curve(
        model, X_train, y_train,
        train_sizes=train_sizes,
        cv=5,
        scoring=auc_scorer,
        n_jobs=-1,
    )
    ax = axes[i]
    ax.fill_between(sizes, tr_scores.mean(1) - tr_scores.std(1),
                    tr_scores.mean(1) + tr_scores.std(1), alpha=0.2, color='#42a5f5')
    ax.fill_between(sizes, val_scores.mean(1) - val_scores.std(1),
                    val_scores.mean(1) + val_scores.std(1), alpha=0.2, color='#ef5350')
    ax.plot(sizes, tr_scores.mean(1),  label='Train AUC', color='#42a5f5', lw=2)
    ax.plot(sizes, val_scores.mean(1), label='Val AUC',   color='#ef5350', lw=2)
    ax.set_xlabel('Training Set Size')
    ax.set_ylabel('ROC-AUC')
    ax.set_title(MODEL_LABELS[name], fontweight='bold')
    ax.legend()
    ax.set_ylim(0.5, 1.05)

plt.suptitle('Learning Curves — ROC-AUC vs Training Size', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
**Notebook complete.** Proceed to `03_xai_deep_dive.ipynb` for SHAP, LIME and counterfactual analysis.